In [ ]:
# =============================================================================
# CELL 1 - COLAB ENVIRONMENT SETUP
# =============================================================================
# Installs, GPU check, Google Drive mount and Hugging Face login.
# facebook/sam3 is a GATED model: you must accept the license at
# https://huggingface.co/facebook/sam3 with the same account as your token.
# =============================================================================

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!pip install -q -U transformers accelerate huggingface_hub supervision opencv-python-headless

import torch
import torchvision

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                :", torch.cuda.get_device_name(0))
    print("GPU memory (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # uses HF_TOKEN from Colab secrets, or paste your token (needs access to facebook/sam3)

print("Environment ready.")


In [ ]:
# =============================================================================
# CELL 2 - IMPORTS
# =============================================================================

import os
import gc
import csv
import time
import json
import hashlib

import numpy as np
import pandas as pd
import torch
import cv2

from PIL import Image
import matplotlib.pyplot as plt

import supervision as sv
from supervision.metrics import MeanAveragePrecision

from transformers import Sam3Model, Sam3Processor

# UAV orthomosaics are very large; disable PIL's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

print("Imports OK. supervision version:", sv.__version__)


In [ ]:
# =============================================================================
# CELL 3 - CONFIGURATION
# =============================================================================
# Every parameter of the study lives here so that a thesis experiment is fully
# described by this single cell.
#
# E02_1 is the NO-TILING experiment: the whole UAV image (resized to MAX_DIM)
# is sent to SAM3 in a single forward pass per anchor, with the exemplar GT
# boxes passed directly as SAM3 "input_boxes" prompts. This is unchanged from
# the original E02_1 pipeline -- only the surrounding structure (config,
# storage, offline NMS, evaluation, metrics) now follows the E02_2 style so
# both experiments are directly comparable.
# =============================================================================

# ------------------------- experiment identity -------------------------------
EXPERIMENT_NAME = "E02_1"     # change per experiment, e.g. "E01", "E02_1", "E02_2"
N_EXEMPLARS     = 3           # 3 = multiple visual prompts, 1 = single visual prompt
USE_TILING      = False       # E02_1 is always whole-image, no tiling
PROMPT_TYPE = "multiple" if N_EXEMPLARS > 1 else "single"

# ------------------------------- dataset -------------------------------------
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0
VALID_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ------------------------------- outputs -------------------------------------
RESULTS_ROOT = f"/content/drive/MyDrive/master_thesis/results_new1/{EXPERIMENT_NAME}"

# --------------------------- whole-image resize -------------------------------
MAX_DIM = 1024                 # resize cap for SAM3 input (E02_1, no tiling)

# --------------------------- SAM3 inference ----------------------------------
# SAM3 is executed EXACTLY ONCE per (image_ID, anchor_idx) at this minimum score.
# Every higher "operating" confidence threshold is applied offline afterwards.
SAM3_INFERENCE_THRESHOLD = 0.30
MASK_THRESHOLD = 0.40          # SAM3 mask binarisation (kept as in the E02_1 code)
KEEP_MASKS = False              # masks are NEVER stored (RAM / GPU / disk / CSV / NPZ)

# ------------------------------ evaluation ------------------------------------
EVAL_IOU_THRESHOLD = 0.50     # IoU needed for a prediction to count as a TP
PROMPT_IGNORE_IOU  = 0.50     # held_out mode: an unmatched prediction whose best IoU
                              # with a PROMPT GT box is >= this value is IGNORED
                              # (neither TP nor FP). Same value as EVAL_IOU_THRESHOLD
                              # so a single IoU threshold governs the whole protocol.

# ------------------------ operating point (frozen) -----------------------------
# No offline confidence x NMS sweep is performed. Both values are fixed from the
# start and used directly wherever an operating point is needed -- same frozen
# values as E02_2, so the two experiments are evaluated on equal footing.
BEST_CONFIDENCE = 0.40
BEST_NMS_IOU = 0.40

EVALUATION_MODES = ["all_gt", "held_out"]
# all_gt   : every GT box of the image is evaluated (classical evaluation).
# held_out : the GT instances that were used as visual prompts are IGNORED, and so
#            are the predictions that fall on them. Answers "how well does SAM3 find
#            the REMAINING Rumex plants after being shown a few examples?".

print(f"Configuration loaded for experiment '{EXPERIMENT_NAME}'")
print(f"  prompts        : {N_EXEMPLARS} ({PROMPT_TYPE})")
print(f"  tiling         : {USE_TILING} (whole image resized to MAX_DIM={MAX_DIM})")
print(f"  SAM3 threshold : {SAM3_INFERENCE_THRESHOLD} (single inference pass per anchor)")
print(f"  mask threshold : {MASK_THRESHOLD}")
print(f"  masks stored   : {KEEP_MASKS}")
print(f"  operating pt   : confidence={BEST_CONFIDENCE}, NMS IoU={BEST_NMS_IOU} (frozen, no sweep)")


In [ ]:
# =============================================================================
# CELL 4 - OUTPUT FOLDERS
# =============================================================================
# results_new1/<EXPERIMENT_NAME>/
#   raw_detections/      pre-NMS detections (NPZ, one file per image x anchor)
#   metrics/             run / image / experiment / dataset level CSVs
#   confusion_matrices/  CSV + PNG for all_gt and held_out
# =============================================================================

RAW_DETECTIONS_DIR    = os.path.join(RESULTS_ROOT, "raw_detections")
METRICS_DIR           = os.path.join(RESULTS_ROOT, "metrics")
CONFUSION_MATRIX_DIR  = os.path.join(RESULTS_ROOT, "confusion_matrices")

for d in [RESULTS_ROOT, RAW_DETECTIONS_DIR, METRICS_DIR,
          CONFUSION_MATRIX_DIR]:
    os.makedirs(d, exist_ok=True)

# Manifest of finished inference runs -> used for crash-safe resuming.
RUN_MANIFEST_CSV = os.path.join(RAW_DETECTIONS_DIR, "runs_manifest.csv")
MANIFEST_COLUMNS = [
    "experiment_name", "image_ID", "anchor_idx", "Prompt_ID", "Prompt_Type",
    "n_gt", "n_prompt_gt", "n_detections_pre_nms",
    "image_width", "image_height", "npz_file", "inference_seconds",
]

print("Output folders ready under:", RESULTS_ROOT)


In [ ]:
# =============================================================================
# CELL 5 - SAM3 MODEL + PROCESSOR
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

sam3_model = Sam3Model.from_pretrained("facebook/sam3", device_map="auto")
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("SAM3 loaded.")
print("  model device:", next(sam3_model.parameters()).device)
print("  model dtype :", next(sam3_model.parameters()).dtype)


In [ ]:
# =============================================================================
# CELL 6 - STABLE REPRODUCIBILITY HELPERS
# =============================================================================
# WHY THIS EXISTS
# The original E02_1 code drew exemplars from a single global
# np.random.default_rng(42) that was advanced once per (image, anchor) in loop
# order. That only stays reproducible if the loop is replayed from scratch in
# EXACTLY the same order every time -- restarting the runtime midway, changing
# the image list, or reordering folders would silently change which exemplars
# are sampled for a given image/anchor.
#
# FIX (same as E02_2): derive the seed from a SHA-256 digest of a plain text
# key. SHA-256 is a fixed mathematical function, so the same key always yields
# the same seed on every machine, every Python version and every session --
# independent of loop order or resume state.
# =============================================================================

def stable_seed(*parts) -> int:
    """
    Deterministic 32-bit seed from any set of values.

    Input : any number of values (strings / ints) that identify the run,
            e.g. stable_seed(EXPERIMENT_NAME, image_id, anchor_idx)
    Output: int in [0, 2**32) - identical in every Python process, forever.
    """
    key = "|".join(str(p) for p in parts)
    digest = hashlib.sha256(key.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % (2 ** 32)


def select_exemplar_indices(n_gt: int,
                            anchor_idx: int,
                            n_exemplars: int,
                            image_id: str,
                            experiment_name: str = EXPERIMENT_NAME) -> list:
    """
    Choose which GT instances of ONE image are used as visual prompts.

    Input : n_gt        - number of GT boxes in the image
            anchor_idx  - index of the GT box this run is "about" (always a prompt)
            n_exemplars - how many prompts in total (1 or 3)
            image_id    - "<folder>/<image name>"
    Output: list of GT indices, ANCHOR FIRST, then the randomly sampled others.

    The anchor is always included; the remaining (n_exemplars - 1) slots are
    filled by sampling without replacement from the other GT boxes of the SAME
    image. If the image does not contain enough other boxes, fewer prompts are
    used (no duplication, no crash).
    """
    seed = stable_seed(experiment_name, image_id, anchor_idx)
    rng = np.random.default_rng(seed)

    others = [i for i in range(n_gt) if i != anchor_idx]
    n_needed = min(n_exemplars - 1, len(others))
    if n_needed > 0:
        chosen = [int(i) for i in rng.choice(others, size=n_needed, replace=False)]
    else:
        chosen = []
    return [int(anchor_idx)] + chosen


def format_prompt_id(exemplar_indices: list) -> str:
    """
    Human-readable id of a prompt set.
      single   -> "5"
      multiple -> "5+12+3"  (the ANCHOR is always the first number)
    """
    return "+".join(str(int(i)) for i in exemplar_indices)


# --- quick self-test: the same key must always give the same seed -------------
_demo = stable_seed(EXPERIMENT_NAME, "folderA/img_001", 3)
print("stable_seed demo :", _demo, "(identical in every session / machine)")
print("exemplar demo    :", select_exemplar_indices(10, 3, N_EXEMPLARS, "folderA/img_001"))


In [ ]:
# =============================================================================
# CELL 7 - DATASET AND YOLO ANNOTATION HELPERS
# =============================================================================
# Unchanged logic from the original E02_1 notebook (it worked); only wrapped
# into functions and given comments, same as E02_2's CELL 7.
# =============================================================================

def load_yolo_boxes(label_path, img_width, img_height, class_id=RUMEX_CLASS_ID):
    """
    Read a YOLO .txt annotation file and convert it to pixel corner boxes.

    Input : label_path            - YOLO txt file
            img_width, img_height - size of the ORIGINAL image in pixels
            class_id              - keep only this class (0 = Rumex)
    Output: np.ndarray (N, 4) float32, boxes as [x1, y1, x2, y2] in pixels.

    YOLO stores normalised (class, x_center, y_center, width, height).
    """
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue                      # skip empty lines
            if int(parts[0]) != class_id:
                continue                      # keep only the requested class
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc = xc * img_width, yc * img_height
            bw, bh = bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def find_label_path(image_filename_no_ext, folder_name):
    """
    Locate the YOLO txt belonging to an image, supporting both a mirrored folder
    structure (labels/<folder>/<n>.txt) and a flat one (labels/<n>.txt).
    Returns the path or None.
    """
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None


def discover_images():
    """
    Walk IMAGES_ROOT and collect every image together with its label file.
    Output: list of (folder, image_path, label_path, image_id) tuples,
            where image_id = "<folder>/<image name without extension>".
    """
    records = []
    for folder in sorted(os.listdir(IMAGES_ROOT)):
        folder_path = os.path.join(IMAGES_ROOT, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(VALID_IMAGE_EXTENSIONS):
                continue
            name_no_ext = os.path.splitext(fname)[0]
            label_path = find_label_path(name_no_ext, folder)
            image_id = f"{folder}/{name_no_ext}"
            records.append((folder, os.path.join(folder_path, fname), label_path, image_id))
    return records


def safe_filename(image_id: str) -> str:
    """'folder/name' -> 'folder__name' so it can be used inside a file name."""
    return image_id.replace("/", "__").replace(os.sep, "__")


image_records = discover_images()
missing_labels = [r for r in image_records if r[2] is None]
valid_images = [r for r in image_records if r[2] is not None]

print(f"Discovered {len(image_records)} images in "
      f"{len(set(r[0] for r in image_records))} folders.")
print(f"With labels: {len(valid_images)} | without labels (skipped): {len(missing_labels)}")
if missing_labels:
    print("  first missing:", [r[3] for r in missing_labels[:5]])


In [ ]:
# =============================================================================
# CELL 8 - IoU
# =============================================================================
# Unchanged from the original notebook - it was already correct and vectorised.
# =============================================================================

def compute_iou_matrix(boxes1, boxes2):
    """
    Pairwise IoU between two sets of [x1, y1, x2, y2] boxes.

    Input : boxes1 (N, 4), boxes2 (M, 4)
    Output: (N, M) matrix, entry [i, j] = IoU(boxes1[i], boxes2[j])
    """
    boxes1 = np.asarray(boxes1, dtype=np.float32).reshape(-1, 4)
    boxes2 = np.asarray(boxes2, dtype=np.float32).reshape(-1, 4)
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])   # left
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])   # top
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])   # right
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])   # bottom

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)


print("IoU helpers ready.")


In [ ]:
# =============================================================================
# CELL 9 - CORRECT ONE-TO-ONE MATCHING
# =============================================================================
# THE BUG IN THE OLD CODE (both E02_1 and E02_2 originally shared this bug)
# The old matcher did:
#     best_gt = argmax(IoU[pred])          <- global best, ignoring availability
#     if best_iou >= thr and best_gt free: match
# So if the globally best GT was already taken, the prediction was thrown away
# even when ANOTHER, still free GT had IoU >= threshold. That under-counted TPs
# (recall too low, false positives too high) on images with clustered plants.
#
# THE CORRECT RULE (implemented here, same as E02_2)
#   sort predictions by confidence, high -> low
#   for each prediction:
#       look ONLY at GT boxes that are still unmatched
#       take the unmatched GT with the highest IoU
#       if that IoU >= evaluation IoU threshold -> match, else leave unmatched
#   one GT can be matched by at most one prediction, and vice versa.
#
# This single function is used for TP / FP / FN / precision / recall / F1 /
# IoU1 / IoU2 / confusion matrices, so the whole thesis uses one definition.
# =============================================================================

def match_one_to_one(pred_boxes, pred_scores, gt_boxes, iou_threshold=EVAL_IOU_THRESHOLD):
    """
    Input : pred_boxes  (P, 4), pred_scores (P,), gt_boxes (G, 4), iou_threshold
    Output: dict with
        'pred_match_gt' (P,) int   - matched GT index per prediction, -1 if unmatched
        'gt_match_pred' (G,) int   - matched prediction index per GT,  -1 if unmatched
        'pred_iou'      (P,) float - IoU of the accepted match, 0.0 if unmatched
        'matched_ious'  list       - IoU values of all accepted matches
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    n_pred, n_gt = len(pred_boxes), len(gt_boxes)

    pred_match_gt = np.full(n_pred, -1, dtype=np.int64)
    gt_match_pred = np.full(n_gt, -1, dtype=np.int64)
    pred_iou = np.zeros(n_pred, dtype=np.float32)  # IoU of the accepted match for every prediction

    if n_pred == 0 or n_gt == 0:
        return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
                "pred_iou": pred_iou, "matched_ious": []}

    iou = compute_iou_matrix(pred_boxes, gt_boxes)
    gt_free = np.ones(n_gt, dtype=bool)  # track which GTs are still available

    # 'stable' keeps the original order for equal scores -> fully deterministic.
    order = np.argsort(-np.asarray(pred_scores, dtype=np.float32), kind="stable")  # confidence, high -> low

    matched_ious = []
    for p in order:
        if not gt_free.any():
            break                                   # nothing left to match
        # Consider ONLY currently unmatched GT boxes (this is the fix).
        candidate_ious = np.where(gt_free, iou[p], -1.0)
        g = int(np.argmax(candidate_ious))  # best FREE GT
        if candidate_ious[g] >= iou_threshold:
            gt_free[g] = False
            pred_match_gt[p] = g
            gt_match_pred[g] = p
            pred_iou[p] = candidate_ious[g]
            matched_ious.append(float(candidate_ious[g]))

    return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
            "pred_iou": pred_iou, "matched_ious": matched_ious}


def safe_f1(precision, recall):
    """F1 = 2PR/(P+R) with a safe zero denominator (returns 0.0)."""
    denom = precision + recall
    return float(2.0 * precision * recall / denom) if denom > 0 else 0.0


print("One-to-one matcher ready (unmatched-GT-aware).")


In [ ]:
# =============================================================================
# CELL 10 - RESIZE HELPER (whole-image, no tiling)
# =============================================================================
# E02_1 sends the WHOLE image to SAM3 in one pass (no tiles). Very large UAV
# orthomosaics are downscaled to MAX_DIM on the longest side purely for SAM3
# input size / speed; predictions are rescaled back to full resolution
# immediately after inference (see CELL 11). Unchanged from the original
# E02_1 code.
# =============================================================================

def resize_for_sam3(img, max_dim=MAX_DIM):
    w, h = img.size
    scale = max_dim / max(w, h)
    if scale >= 1:
        return img, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    return img.resize((new_w, new_h), Image.BILINEAR), scale


print(f"Resize helper ready (MAX_DIM={MAX_DIM}).")


In [ ]:
# =============================================================================
# CELL 11 - SAM3 WHOLE-IMAGE INFERENCE (no tiling, exemplar boxes as prompts)
# =============================================================================
# This is the E02_1 method, KEPT AS IT IS:
#   - resize whole image to MAX_DIM
#   - scale exemplar GT boxes into that resized space
#   - single SAM3 forward pass, no tiling, no exemplar-strip composition
#     (the exemplar plants are already visible in the same image, so they can
#     be passed directly as positive box prompts -- unlike E02_2, which needs
#     the strip trick because a tile may not contain the exemplar plant)
#   - scale predicted boxes back up to full-res
#
# SAM3 is called ONCE per (image_ID, anchor_idx) at SAM3_INFERENCE_THRESHOLD;
# everything above that (operating confidence, NMS) is applied offline, same
# protocol as E02_2.
# =============================================================================

def _to_numpy(x):
    """Handles tensors (incl. bf16/fp16), lists of tensors, or plain arrays."""
    if torch.is_tensor(x):
        if x.dtype in (torch.bfloat16, torch.float16):
            x = x.float()
        return x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        if len(x) > 0 and torch.is_tensor(x[0]):
            x = [t.float() if t.dtype in (torch.bfloat16, torch.float16) else t for t in x]
            return torch.stack([t.detach().cpu() for t in x]).numpy()
        return np.array(x)
    return np.array(x)


def run_sam3_whole_image(image: Image.Image, exemplar_boxes_fullres: list,
                         threshold: float = SAM3_INFERENCE_THRESHOLD,
                         mask_threshold: float = MASK_THRESHOLD):
    """
    exemplar_boxes_fullres: list of [x1,y1,x2,y2] in the ORIGINAL image's pixel coords.

    Returns: pred_boxes_fullres (np.ndarray Nx4), pred_scores (np.ndarray N),
             all boxes/scores with score >= threshold. Masks are requested from
             SAM3 internally (needed by post-processing) but never stored.
    """
    img_w, img_h = image.size
    image_sam, sam_scale = resize_for_sam3(image, MAX_DIM)

    input_boxes_xyxy = [[c * sam_scale for c in box] for box in exemplar_boxes_fullres]
    input_boxes = [input_boxes_xyxy]
    input_boxes_labels = [[1] * len(exemplar_boxes_fullres)]  # 1 = positive prompt

    inputs = sam3_processor(
        images=image_sam,
        input_boxes=input_boxes,
        input_boxes_labels=input_boxes_labels,
        return_tensors="pt",
    ).to(sam3_model.device)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=mask_threshold,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]

    pred_boxes_np = _to_numpy(results["boxes"]).reshape(-1, 4)
    pred_scores_np = _to_numpy(results["scores"]).reshape(-1)

    # Boxes came back in image_sam's (resized) coordinate space -> rescale to full-res
    if len(pred_boxes_np) > 0:
        pred_boxes_np = pred_boxes_np / sam_scale

    # masks are never kept (KEEP_MASKS = False) -> drop the reference immediately
    del results, outputs, inputs

    return pred_boxes_np, pred_scores_np


print("SAM3 whole-image inference helper ready (no tiling, direct box prompts).")


In [ ]:
# =============================================================================
# CELL 12 - PRE-NMS DETECTION STORAGE
# =============================================================================
# WHAT IS SAVED AND WHY
# For every run (= one image x one anchor) we store the detections AFTER
#   SAM3 inference at SAM3_INFERENCE_THRESHOLD, converted to original-image
#   coordinates, but BEFORE any operating confidence threshold and BEFORE NMS.
#
# That is exactly the state needed to replay any (confidence, NMS IoU) pair
# offline without ever running SAM3 again. Masks are never stored.
# NPZ is used because it is compact and loads fast; the metric tables are CSV.
# Same protocol as E02_2, minus the tile bookkeeping which does not apply here.
# =============================================================================

def run_npz_path(image_id, anchor_idx):
    """Path of the NPZ holding the pre-NMS detections of one run."""
    return os.path.join(RAW_DETECTIONS_DIR,
                        f"{safe_filename(image_id)}__anchor{int(anchor_idx):03d}.npz")


def save_run_detections(image_id, anchor_idx, boxes, scores, gt_boxes,
                        prompt_indices, image_size):
    """
    Write one run's pre-NMS detections to NPZ. The file is self-contained: it also
    stores the GT boxes and the prompt indices, so the whole offline evaluation
    can run without re-opening images or label files.
    """
    path = run_npz_path(image_id, anchor_idx)
    np.savez_compressed(
        path,
        experiment_name=np.array(EXPERIMENT_NAME),
        image_ID=np.array(image_id),
        anchor_idx=np.array(int(anchor_idx)),
        prompt_indices=np.array(prompt_indices, dtype=np.int32),
        image_width=np.array(int(image_size[0])),
        image_height=np.array(int(image_size[1])),
        gt_boxes=gt_boxes.astype(np.float32),
        boxes=boxes.astype(np.float32),      # x1,y1,x2,y2 (original image coords)
        scores=scores.astype(np.float32),    # confidence >= SAM3_INFERENCE_THRESHOLD
    )
    return path


def load_run_detections(path):
    """Read one run NPZ back into a plain python dict."""
    with np.load(path, allow_pickle=False) as z:
        return {
            "image_ID": str(z["image_ID"]),
            "anchor_idx": int(z["anchor_idx"]),
            "prompt_indices": z["prompt_indices"].astype(int),
            "image_width": int(z["image_width"]),
            "image_height": int(z["image_height"]),
            "gt_boxes": z["gt_boxes"].reshape(-1, 4),
            "boxes": z["boxes"].reshape(-1, 4),
            "scores": z["scores"].reshape(-1),
        }


print("Pre-NMS detection storage ready ->", RAW_DETECTIONS_DIR)


In [ ]:
# =============================================================================
# CELL 13 - MAIN GPU INFERENCE LOOP
# =============================================================================
# FOR EACH IMAGE:
#     open the original image ONCE
#     read its GT boxes ONCE
#     FOR EACH anchor (= every GT box, once):
#         select the exemplars deterministically (stable_seed, CELL 6)
#         run SAM3 ONCE over the whole (resized) image, boxes as direct prompts
#         save the PRE-NMS detections (NPZ)
#     release the image
#
# NO confidence sweep, NO NMS sweep and NO metric computation happens here -
# that is all done offline in the following cells, same protocol as E02_2.
# The loop is resumable: finished runs are listed in runs_manifest.csv.
# =============================================================================

# ---- resume support ---------------------------------------------------------
done_runs = set()
manifest_exists = os.path.exists(RUN_MANIFEST_CSV)
if manifest_exists:
    _m = pd.read_csv(RUN_MANIFEST_CSV)
    _m = _m[_m["experiment_name"] == EXPERIMENT_NAME]
    done_runs = set(zip(_m["image_ID"].astype(str), _m["anchor_idx"].astype(int)))
    print(f"Resuming: {len(done_runs)} runs already finished for {EXPERIMENT_NAME}.")

manifest_file = open(RUN_MANIFEST_CSV, "a", newline="")
manifest_writer = csv.DictWriter(manifest_file, fieldnames=MANIFEST_COLUMNS)
if not manifest_exists:
    manifest_writer.writeheader()

start_time = time.time()
n_new_runs = 0
image_times = []
n_total_images = len(valid_images)

for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
    image_t0 = time.time()

    # ---------------- open the original image exactly once --------------------
    image = Image.open(image_path).convert("RGB")
    img_w, img_h = image.size
    gt_boxes = load_yolo_boxes(label_path, img_w, img_h, RUMEX_CLASS_ID)
    n_gt = len(gt_boxes)

    if n_gt == 0:
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
        image.close(); del image; gc.collect()
        continue

    # skip the whole image if every anchor is already done
    if all((image_id, a) in done_runs for a in range(n_gt)):
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: all "
              f"{n_gt} anchors already done, skipped.")
        image.close(); del image; gc.collect()
        continue

    for anchor_idx in range(n_gt):                 # every GT box is an anchor once
        if (image_id, anchor_idx) in done_runs:
            continue
        run_t0 = time.time()

        # deterministic prompt selection (SHA-256 based, see CELL 6)
        exemplar_indices = select_exemplar_indices(n_gt, anchor_idx, N_EXEMPLARS, image_id)
        prompt_id = format_prompt_id(exemplar_indices)
        exemplar_boxes_fullres = [gt_boxes[i].tolist() for i in exemplar_indices]

        pred_boxes_np, pred_scores_np = run_sam3_whole_image(
            image, exemplar_boxes_fullres,
            threshold=SAM3_INFERENCE_THRESHOLD, mask_threshold=MASK_THRESHOLD,
        )

        npz_path = save_run_detections(image_id, anchor_idx, pred_boxes_np, pred_scores_np,
                                       gt_boxes, exemplar_indices, (img_w, img_h))

        run_seconds = time.time() - run_t0
        manifest_writer.writerow({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": image_id,
            "anchor_idx": anchor_idx,
            "Prompt_ID": prompt_id,
            "Prompt_Type": PROMPT_TYPE,
            "n_gt": n_gt,
            "n_prompt_gt": len(exemplar_indices),
            "n_detections_pre_nms": int(len(pred_scores_np)),
            "image_width": img_w,
            "image_height": img_h,
            "npz_file": os.path.basename(npz_path),
            "inference_seconds": round(run_seconds, 2),
        })
        manifest_file.flush()
        n_new_runs += 1

        print(f"  [{EXPERIMENT_NAME}] run #{n_new_runs} | {image_id} | "
              f"anchor={anchor_idx} ({anchor_idx + 1}/{n_gt}) | prompt={prompt_id} | "
              f"pre-NMS detections={len(pred_scores_np)} | {run_seconds:.1f}s")

        del pred_boxes_np, pred_scores_np
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    # ---------------- release the image -----------------------------------
    image.close()
    del image
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    image_elapsed = time.time() - image_t0
    image_times.append(image_elapsed)
    avg_per_image = float(np.mean(image_times))
    eta = (n_total_images - img_idx) * avg_per_image
    print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id} done | "
          f"{n_gt} GT box(es) | {image_elapsed:.1f}s | avg/image={avg_per_image:.1f}s | "
          f"ETA={eta / 60:.1f} min ({eta / 3600:.2f} h)")

manifest_file.close()
total_elapsed = time.time() - start_time
print(f"\nInference finished for {EXPERIMENT_NAME}: {n_new_runs} new runs.")
print(f"Total time: {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.2f} h)")
print(f"Pre-NMS detections in: {RAW_DETECTIONS_DIR}")


In [ ]:
# =============================================================================
# CELL 14 - LOAD CACHED PRE-NMS DETECTIONS
# =============================================================================
# From here on SAM3 is never touched again. Everything below works on the NPZ
# files written in CELL 13, so you can restart the runtime, free the GPU and
# still redo the complete evaluation in a few minutes.
# =============================================================================

manifest = pd.read_csv(RUN_MANIFEST_CSV)
manifest = manifest[manifest["experiment_name"] == EXPERIMENT_NAME].copy()
manifest = manifest.drop_duplicates(subset=["image_ID", "anchor_idx"], keep="last")

runs = []
for _, row in manifest.iterrows():
    path = os.path.join(RAW_DETECTIONS_DIR, str(row["npz_file"]))
    if not os.path.exists(path):
        print("MISSING npz (skipped):", path)
        continue
    run = load_run_detections(path)
    run["Prompt_ID"] = str(row["Prompt_ID"])
    run["Prompt_Type"] = str(row["Prompt_Type"])
    runs.append(run)

print(f"Loaded {len(runs)} runs "
      f"({manifest['image_ID'].nunique()} images) for {EXPERIMENT_NAME}.")
print("Total pre-NMS detections:", int(sum(len(r['scores']) for r in runs)))
print("Total GT boxes over all runs:", int(sum(len(r['gt_boxes']) for r in runs)))


In [ ]:
# =============================================================================
# CELL 15 - OFFLINE NMS
# =============================================================================
# Even without tiling, SAM3 can occasionally propose more than one overlapping
# box for the same plant in a single forward pass. NMS keeps the highest-
# scoring box of each overlapping group. The NMS IoU threshold is a TUNABLE
# parameter applied offline, so the raw pre-NMS detections stay untouched on
# disk and every value can be replayed -- same protocol as E02_2 (no tile
# provenance here since there are no tiles to attribute duplicates to).
# =============================================================================

def nms(boxes, scores, iou_threshold):
    """
    Input : boxes (N,4), scores (N,), iou_threshold
    Output: keep - list of kept indices, highest score first.
    A detection is suppressed when its IoU with an already kept, higher-scoring
    detection is GREATER than the threshold.
    """
    n = len(boxes)
    if n == 0:
        return []
    order = list(np.argsort(-np.asarray(scores, dtype=np.float32), kind="stable"))
    keep = []
    while order:
        i = int(order[0])
        keep.append(i)
        rest = np.array(order[1:], dtype=int)
        if rest.size == 0:
            break
        ious = compute_iou_matrix(boxes[i:i + 1], boxes[rest])[0]
        order = list(rest[ious <= iou_threshold])
    return keep


def apply_nms_to_run(run, iou_threshold):
    """
    Apply NMS to one run's pre-NMS detections.

    Output: dict with 'boxes', 'scores' of the surviving detections SORTED BY
    SCORE (high -> low). Sorting by score means that applying an operating
    confidence threshold later is just a prefix selection.
    """
    keep = np.array(nms(run["boxes"], run["scores"], iou_threshold), dtype=int)
    return {
        "boxes": run["boxes"][keep].reshape(-1, 4),
        "scores": run["scores"][keep].reshape(-1),
        "n_pre_nms": int(len(run["scores"])),
    }


print("Offline NMS ready. Threshold used (frozen):", BEST_NMS_IOU)


In [ ]:
# =============================================================================
# CELL 16 - EVALUATION CORE: all_gt AND held_out
# =============================================================================
# all_gt   : classical evaluation, every GT box of the image counts.
#
# held_out : the GT instances that were shown to SAM3 as visual prompts are
#            REMOVED from the GT set, and predictions that fall on those prompt
#            plants are IGNORED (they are neither TP nor FP, they simply do not
#            exist for this evaluation). It answers: "after being shown a few
#            examples, how well does SAM3 find the REMAINING Rumex plants?"
#
# ORDER OF OPERATIONS (important, this is the agreed protocol):
#   1. match predictions to the evaluated (non-prompt) GT with the corrected
#      one-to-one matcher at EVAL_IOU_THRESHOLD = 0.50
#   2. every STILL UNMATCHED prediction whose best IoU with a PROMPT GT box is
#      >= PROMPT_IGNORE_IOU (0.50) becomes IGNORED
#   3. whatever is still unmatched is a false positive
#   Prompt GT boxes themselves are never counted as false negatives.
# =============================================================================

STATUS_FP, STATUS_TP, STATUS_IGNORED = 0, 1, 2


def split_gt_for_mode(gt_boxes, prompt_indices, mode):
    """
    Input : all GT boxes of the image, the indices used as visual prompts, mode
    Output: (evaluated_gt_boxes, prompt_gt_boxes)
            all_gt   -> (all boxes, empty)
            held_out -> (non-prompt boxes, prompt boxes)
    """
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    if mode == "all_gt":
        return gt_boxes, np.zeros((0, 4), dtype=np.float32)
    is_prompt = np.zeros(len(gt_boxes), dtype=bool)
    prompt_indices = np.asarray(prompt_indices, dtype=int)
    if len(prompt_indices):
        is_prompt[prompt_indices] = True
    return gt_boxes[~is_prompt], gt_boxes[is_prompt]


def evaluate_run_predictions(pred_boxes, pred_scores, eval_gt_boxes, prompt_gt_boxes,
                             eval_iou=EVAL_IOU_THRESHOLD, ignore_iou=PROMPT_IGNORE_IOU):
    """
    Evaluate ONE prediction set against ONE GT set.

    Output dict:
      status      (P,) int  - STATUS_TP / STATUS_FP / STATUS_IGNORED per prediction
      TP, FP, FN, n_ignored, n_eval_gt, n_pred
      precision, recall, F1
      IoU1 - mean IoU of the MATCHED prediction/GT pairs only
             ("when it finds a plant, how well is it localised?")
      IoU2 - sum of matched IoUs divided by the number of evaluated GT boxes
             ("localisation quality over ALL plants, missed ones count as 0")
      valid_for_macro - False when there is no GT left to evaluate (held_out runs
             in which every plant of the image was used as a prompt)
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(pred_scores, dtype=np.float32).reshape(-1)
    eval_gt_boxes = np.asarray(eval_gt_boxes, dtype=np.float32).reshape(-1, 4)
    prompt_gt_boxes = np.asarray(prompt_gt_boxes, dtype=np.float32).reshape(-1, 4)

    n_pred, n_eval_gt = len(pred_boxes), len(eval_gt_boxes)
    status = np.full(n_pred, STATUS_FP, dtype=np.int8) # start eveything as FP then valid matches TP then leftover predictions on prompt as ignored and everything still FP

    # step 1 - corrected one-to-one matching against the evaluated GT
    match = match_one_to_one(pred_boxes, pred_scores, eval_gt_boxes, eval_iou)
    status[match["pred_match_gt"] >= 0] = STATUS_TP # Mark matched predictions as TP

    # step 2 - ignore the leftovers that sit on a PROMPT plant
    # Therefore prompt-ignore logic applies only to predictions that failed to match evaluated GT
    if n_pred and len(prompt_gt_boxes):
        leftover = np.where(status == STATUS_FP)[0] # Find predictions that are still FP (unmatched preds)
        if len(leftover):
            best_prompt_iou = compute_iou_matrix(pred_boxes[leftover], prompt_gt_boxes).max(axis=1)
            status[leftover[best_prompt_iou >= ignore_iou]] = STATUS_IGNORED

    tp = int((status == STATUS_TP).sum())
    fp = int((status == STATUS_FP).sum())          # step 3 - the rest are FP
    n_ignored = int((status == STATUS_IGNORED).sum())
    fn = int(n_eval_gt - tp)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / n_eval_gt if n_eval_gt > 0 else 0.0
    f1 = safe_f1(precision, recall)
    matched_ious = match["matched_ious"]
    iou1 = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2 = float(np.sum(matched_ious) / n_eval_gt) if n_eval_gt > 0 else 0.0

    return {
        "status": status, "pred_match_gt": match["pred_match_gt"],
        "TP": tp, "FP": fp, "FN": fn, "n_ignored": n_ignored,
        "n_eval_gt": n_eval_gt, "n_pred": n_pred,
        "precision": float(precision), "recall": float(recall), "F1": float(f1),
        "IoU1": iou1, "IoU2": iou2,
        "valid_for_macro": bool(n_eval_gt > 0),
    }


print("Evaluation core ready (modes:", EVALUATION_MODES, ")")

In [ ]:
# =============================================================================
# CELL 17 - AP50 AND AP50:95  (supervision.metrics.MeanAveragePrecision)
# =============================================================================
# WHY AP IS COMPUTED DIFFERENTLY FROM PRECISION / RECALL / F1
#
#   AP is an area under the precision-recall curve. That curve is produced by
#   walking through ALL detections ordered by confidence. Truncating the
#   detection list at an operating threshold would simply cut the tail off the
#   curve and report a smaller area - which says nothing about model quality.
#   Therefore AP always uses ALL saved predictions with score >= 0.30 (the SAM3
#   inference threshold), after the selected NMS.
#
#   Precision / recall / F1 / IoU1 / IoU2 describe ONE operating point: they
#   answer "if I deploy the detector with confidence >= c, what happens?".
#   Those use only the detections that survive the selected operating threshold.
#
#   Both numbers can appear in the same row - they answer different questions.
# =============================================================================

# just a helper function that converts NumPy boxes into the format expected by Supervision
def make_detections(boxes, scores=None):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    n = len(boxes)
    if scores is None:
        return sv.Detections(xyxy=boxes, class_id=np.zeros(n, dtype=int))
    return sv.Detections(xyxy=boxes,
                         confidence=np.asarray(scores, dtype=np.float32).reshape(-1),
                         class_id=np.zeros(n, dtype=int))


def compute_ap(pred_list, gt_list):
    """
    Input : two equally long lists of supervision Detections (predictions / GT).
            One entry = one evaluation episode (one run).
    Output: (AP50, AP50_95). NaN when there is no GT at all to evaluate.
    Passing several episodes at once gives the POOLED (dataset-level) AP, in which
    the detections of all episodes are ranked together in one PR curve.
    """
    if len(gt_list) == 0 or sum(len(g) for g in gt_list) == 0: # Check whether there is any GT
        return float("nan"), float("nan")
    try:
        # Create the Supervision metric (metric calculator, provides all pred and GTs, calculates the AP results)
        # Supervision exposes map50 for IoU=0.50 and map50_95 for IoUs 0.50:0.95
        result = MeanAveragePrecision().update(pred_list, gt_list).compute()
        ap50, ap5095 = float(result.map50), float(result.map50_95)
        # supervision returns -1.0 when a metric is undefined -> report NaN instead,
        # so it is excluded from means instead of dragging them down.
        return (ap50 if ap50 >= 0 else float("nan"),
                ap5095 if ap5095 >= 0 else float("nan"))
    except Exception as e:
        print("   (AP computation failed:", e, ")")
        return float("nan"), float("nan")


# Its job is not to calculate AP yet, Its job is to prepare: pred and GT for one run so they can later be given to compute_ap(...)
# nms_run: contains your detections after NMS
def ap_inputs_for_run(nms_run, eval_gt, prompt_gt):
    """
    Build the (prediction, GT) episode used for AP of ONE run.
    All post-NMS predictions with score >= SAM3_INFERENCE_THRESHOLD are used;
    in held_out mode the predictions that were IGNORED (they belong to prompt
    plants) are removed first, exactly like in the operating-point evaluation.
    """
    ev = evaluate_run_predictions(nms_run["boxes"], nms_run["scores"], eval_gt, prompt_gt) # marks each pred as TP,FP,IGNORED, it cares only about the ignored predictions for AP calculation bcz in held_out mode , predictions corresponding to prompt rumex must be removed before AP
    keep = ev["status"] != STATUS_IGNORED  # Remove ignored predictions
    # Convert remaining predictions to Supervision format
    return (make_detections(nms_run["boxes"][keep], nms_run["scores"][keep]),
            make_detections(eval_gt))


print("AP helpers ready (AP always uses every prediction >= "
      f"{SAM3_INFERENCE_THRESHOLD} after NMS).")

In [ ]:
# =============================================================================
# CELL 18 - OPERATING-POINT EVALUATION
# =============================================================================
# Choose one confidence threshold, remove predictions below it, then send the
# remaining predictions to CELL 16 for evaluation. This is the function that
# gives Precision, Recall, F1, IoU1, and IoU2 at one operating threshold.
# =============================================================================

def evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, confidence_threshold):
    """
    Input : nms_run  - output of apply_nms_to_run (sorted by score, high -> low) => contains the detections after NMS            eval_gt / prompt_gt - from split_gt_for_mode
            confidence_threshold - the operating point being tested
    Output: the dict of evaluate_run_predictions for the thresholded predictions.
    """
    keep = nms_run["scores"] >= confidence_threshold # boolean mask where True entries are selected and the False entries are excluded
    return evaluate_run_predictions(nms_run["boxes"][keep], nms_run["scores"][keep],
                                    eval_gt, prompt_gt)


print("Operating-point evaluation ready. Confidence threshold used (frozen):", BEST_CONFIDENCE)

In [ ]:
# =============================================================================
# CELL 19 - OPERATING CONFIGURATION (FROZEN, NO OFFLINE SWEEP)
# =============================================================================
# No offline confidence x NMS sweep is performed. BEST_CONFIDENCE and
# BEST_NMS_IOU are fixed in CELL 3 (both = 0.40, same as E02_2) and are used
# directly, unchanged, by every cell from here on.
# =============================================================================

print("Operating configuration is frozen (no sweep performed):")
print(f"  Confidence threshold = {BEST_CONFIDENCE:.2f}")
print(f"  NMS IoU threshold    = {BEST_NMS_IOU:.2f}")


In [ ]:
# use the best NMS threshold and best confidence threshold to compute the final metrics for every run
# =============================================================================
# CELL 20 - RUN-LEVEL METRICS  (one run = one image x one anchor/prompt set)
# =============================================================================
# Everything is evaluated at the FROZEN configuration from CELL 19.
#   AP50 / AP50_95 : all post-NMS predictions >= 0.30, confidence-ranked
#   P / R / F1 / IoU1 / IoU2 / TP / FP / FN : only predictions >= BEST_CONFIDENCE
#
# Special case (held_out with no evaluable GT, i.e. every plant of the image was
# used as a prompt): the metrics are written as NaN and valid_for_macro = False so
# they are excluded from every mean/std, but TP/FN = 0 and the real FP count are
# kept, because such a run can still produce false positives that must show up in
# the pooled counts and in the confusion matrix.
# =============================================================================

run_rows = []
for run in runs:
    nms_run = apply_nms_to_run(run, BEST_NMS_IOU)
    for mode in EVALUATION_MODES:
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)

        # ---- AP: every prediction >= 0.30 after NMS (ignored ones removed) ----
        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
        ap50, ap5095 = compute_ap([p_det], [g_det])

        # ---- operating point -------------------------------------------------
        ev = evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, BEST_CONFIDENCE)
        valid = ev["valid_for_macro"]
        nan = float("nan")

        run_rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": run["image_ID"],
            "anchor_idx": run["anchor_idx"],
            "Prompt_ID": run["Prompt_ID"],
            "Prompt_Type": run["Prompt_Type"],
            "evaluation_mode": mode,
            "confidence_threshold": BEST_CONFIDENCE,
            "nms_iou_threshold": BEST_NMS_IOU,
            "n_gt_total": int(len(run["gt_boxes"])),
            "n_prompt_gt": int(len(run["prompt_indices"])) if mode == "held_out" else 0,
            "n_eval_gt": ev["n_eval_gt"],
            "n_predictions": ev["n_pred"],
            "n_ignored_predictions": ev["n_ignored"],
            "AP50": ap50 if valid else nan,
            "AP50_95": ap5095 if valid else nan,
            "precision": ev["precision"] if valid else nan,
            "recall": ev["recall"] if valid else nan,
            "F1": ev["F1"] if valid else nan,
            "IoU1": ev["IoU1"] if valid else nan,
            "IoU2": ev["IoU2"] if valid else nan,
            "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
            "valid_for_macro": valid,
        })

run_level_df = pd.DataFrame(run_rows)
RUN_LEVEL_CSV = os.path.join(METRICS_DIR, "run_level_metrics.csv")
run_level_df.to_csv(RUN_LEVEL_CSV, index=False)

print(f"Run-level metrics: {len(run_level_df)} rows -> {RUN_LEVEL_CSV}")
for mode in EVALUATION_MODES:
    sub = run_level_df[run_level_df["evaluation_mode"] == mode]
    print(f"  {mode:9s}: {len(sub)} runs, "
          f"{int(sub['valid_for_macro'].sum())} valid for macro averaging, "
          f"F1_mean={sub['F1'].mean():.4f}")

In [ ]:
# =============================================================================
# CELL 21 - IMAGE-LEVEL METRICS
# =============================================================================
# All anchor runs of the same image are averaged into ONE value per image and per
# evaluation mode. The std here is the spread BETWEEN the different anchor/prompt
# selections of the SAME image, i.e. "how sensitive is the result to which plant
# was used as the visual prompt?".
# NaN rows (held_out runs with no evaluable GT) are ignored by pandas mean/std.
# std is NaN when an image has only one valid run - that is expected.
# =============================================================================

METRIC_COLUMNS = ["AP50", "AP50_95", "precision", "recall", "F1", "IoU1", "IoU2"]

image_rows = []
for (image_id, mode), grp in run_level_df.groupby(["image_ID", "evaluation_mode"]):
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "image_ID": image_id,
        "evaluation_mode": mode,
        "confidence_threshold": BEST_CONFIDENCE,
        "nms_iou_threshold": BEST_NMS_IOU,
        "n_runs_total": int(len(grp)),
        "n_runs_valid_for_macro": int(grp["valid_for_macro"].sum()),
        "TP_sum": int(grp["TP"].sum()),
        "FP_sum": int(grp["FP"].sum()),
        "FN_sum": int(grp["FN"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = grp[col].mean()      # NaNs skipped automatically
        row[f"{col}_std"] = grp[col].std()        # sample std (ddof=1)
    image_rows.append(row)

image_level_df = pd.DataFrame(image_rows).sort_values(
    ["evaluation_mode", "image_ID"]).reset_index(drop=True)
IMAGE_LEVEL_CSV = os.path.join(METRICS_DIR, "image_level_metrics.csv")
image_level_df.to_csv(IMAGE_LEVEL_CSV, index=False)

print(f"Image-level metrics: {len(image_level_df)} rows -> {IMAGE_LEVEL_CSV}")
print(image_level_df.groupby("evaluation_mode")[
    ["AP50_mean", "precision_mean", "recall_mean", "F1_mean", "IoU1_mean", "IoU2_mean"]
].mean().to_string())

In [ ]:
# =============================================================================
# CELL 22 - EXPERIMENT-LEVEL SUMMARY
# =============================================================================
# Computed from the IMAGE-LEVEL values, not from the raw run rows, so that every
# UAV image contributes exactly the same weight regardless of how many GT boxes
# (and therefore how many anchor runs) it contains.
# The std here is the variation BETWEEN UAV images.
# =============================================================================

summary_rows = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_exemplars": N_EXEMPLARS,
        "use_tiling": USE_TILING,
        "max_dim": MAX_DIM,
        "confidence_threshold": BEST_CONFIDENCE,
        "nms_iou_threshold": BEST_NMS_IOU,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
        "n_images": int(sub["image_ID"].nunique()),
        "n_runs": int(sub["n_runs_total"].sum()),
        "n_runs_valid_for_macro": int(sub["n_runs_valid_for_macro"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = sub[f"{col}_mean"].mean()
        row[f"{col}_std"] = sub[f"{col}_mean"].std()   # spread between images
    summary_rows.append(row)

experiment_summary_df = pd.DataFrame(summary_rows)
EXPERIMENT_SUMMARY_CSV = os.path.join(METRICS_DIR, "experiment_summary.csv")
experiment_summary_df.to_csv(EXPERIMENT_SUMMARY_CSV, index=False)

print(f"Experiment summary -> {EXPERIMENT_SUMMARY_CSV}\n")
print(experiment_summary_df.to_string(index=False))


In [ ]:
# =============================================================================
# CELL 23 - POOLED DATASET AP50 / AP50:95
# =============================================================================
# CELL 20 computes AP separately for each run.
# This cell gives all runs to the AP evaluator together and computes one
# overall AP for the experiment (one result row per evaluation mode).
# =============================================================================
# This is NOT the mean of the image-level AP values. All runs are handed to
# supervision as evaluation EPISODES at once, so every detection of the whole
# dataset is ranked in ONE precision-recall curve.
#
# NOTE for the thesis text: because every GT box of an image becomes an anchor
# once, the same UAV image appears in several episodes (once per prompt set).
# The pooled AP is therefore computed over "pooled evaluation episodes", not over
# unique images - it measures the ranking quality of the whole experiment.
# =============================================================================

# At IoU 0.50, each detection is judged as TP or FP against the GT of its own episode
# Then, conceptually, the confidence-ranked detection sequence across the experiment is: ... As detections
# are accumulated in confidence order, overall precision and recall change, producing the experiment-level PR curve

dataset_rows = []
for mode in EVALUATION_MODES:
    pred_list, gt_list, images_used = [], [], set()
    for run in runs:
        nms_run = apply_nms_to_run(run, BEST_NMS_IOU)
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)
        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt) # Prepare this run's AP episode
        pred_list.append(p_det)
        gt_list.append(g_det)
        images_used.add(run["image_ID"])
    ap50, ap5095 = compute_ap(pred_list, gt_list) # After ALL runs, calculate AP (after the for run in runs loop)
    dataset_rows.append({
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "n_images": len(images_used),
        "n_runs": len(runs),
        "confidence_used_for_AP": SAM3_INFERENCE_THRESHOLD,   # AP always uses >= 0.30
        "nms_iou_threshold": BEST_NMS_IOU,
        "dataset_AP50": ap50,
        "dataset_AP50_95": ap5095,
    })
    del pred_list, gt_list
    gc.collect()

dataset_ap_df = pd.DataFrame(dataset_rows)
DATASET_AP_CSV = os.path.join(METRICS_DIR, "dataset_ap_metrics.csv")
dataset_ap_df.to_csv(DATASET_AP_CSV, index=False)

print(f"Dataset pooled AP -> {DATASET_AP_CSV}\n")
print(dataset_ap_df.to_string(index=False))
print("\nFor comparison, the MEAN of the image-level AP50 values "
      "(a different quantity):")
print(image_level_df.groupby("evaluation_mode")["AP50_mean"].mean().to_string())

In [ ]:
# =============================================================================
# CELL 24 - DATASET-LEVEL CONFUSION MATRICES
# =============================================================================
# One class (Rumex) plus a background row/column:
#     Actual Rumex      -> Predicted Rumex      = TP
#     Actual Rumex      -> Predicted Background = FN  (missed plants)
#     Actual Background -> Predicted Rumex      = FP  (spurious detections)
#     Actual Background -> Predicted Background = not defined for detection
#                                                 (there are no true negatives)
# Counts are pooled over every run at the frozen configuration. In held_out mode
# the prompt plants and the detections that were ignored do not appear anywhere.
# =============================================================================

def plot_confusion_matrix(tp, fp, fn, title, png_path):
    """2x2 detection confusion matrix; the background/background cell stays empty."""
    matrix = np.array([[tp, fn], [fp, np.nan]], dtype=float)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(np.nan_to_num(matrix, nan=0.0), cmap="Blues")
    ax.set_xticks([0, 1], ["Predicted\nRumex", "Predicted\nBackground"])
    ax.set_yticks([0, 1], ["Actual\nRumex", "Actual\nBackground"])
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", "n/a\n(no true\nnegatives)"]]
    vmax = np.nanmax(matrix) if np.nanmax(matrix) > 0 else 1.0
    for i in range(2):
        for j in range(2):
            value = matrix[i, j]
            colour = "white" if (not np.isnan(value) and value > 0.5 * vmax) else "black"
            ax.text(j, i, labels[i][j], ha="center", va="center",
                    color=colour, fontsize=11)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(png_path, dpi=200)
    plt.show()
    plt.close(fig)


confusion_summary = []
for mode in EVALUATION_MODES:
    sub = run_level_df[run_level_df["evaluation_mode"] == mode]
    tp, fp, fn = int(sub["TP"].sum()), int(sub["FP"].sum()), int(sub["FN"].sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = safe_f1(precision, recall)

    cm_df = pd.DataFrame(
        [[tp, fn], [fp, np.nan]],
        index=["actual_rumex", "actual_background"],
        columns=["predicted_rumex", "predicted_background"],
    )
    csv_path = os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.csv")
    cm_df.to_csv(csv_path)

    png_path = os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.png")
    plot_confusion_matrix(
        tp, fp, fn,
        f"{EXPERIMENT_NAME} - {mode}\nconf={BEST_CONFIDENCE:.2f}, "
        f"NMS IoU={BEST_NMS_IOU:.2f}, eval IoU={EVAL_IOU_THRESHOLD:.2f}",
        png_path)

    confusion_summary.append({
        "experiment_name": EXPERIMENT_NAME, "evaluation_mode": mode,
        "TP": tp, "FP": fp, "FN": fn,
        "precision_micro": precision, "recall_micro": recall, "F1_micro": f1,
        "confidence_threshold": BEST_CONFIDENCE, "nms_iou_threshold": BEST_NMS_IOU,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
    })
    print(f"{mode:9s}: TP={tp}  FP={fp}  FN={fn}  "
          f"P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")

confusion_summary_df = pd.DataFrame(confusion_summary)
confusion_summary_df.to_csv(
    os.path.join(CONFUSION_MATRIX_DIR, "confusion_matrix_summary.csv"), index=False)
print("\nConfusion matrices saved to:", CONFUSION_MATRIX_DIR)

In [ ]:
# =============================================================================
# CELL 25 - FINAL OUTPUT SUMMARY
# =============================================================================

print("=" * 78)
print(f"EXPERIMENT {EXPERIMENT_NAME} - FINAL SUMMARY")
print("=" * 78)
print(f"Prompts per run          : {N_EXEMPLARS} ({PROMPT_TYPE})")
print(f"Tiling                   : {USE_TILING}  (whole image resized to MAX_DIM={MAX_DIM}px)")
print(f"SAM3 inference threshold : {SAM3_INFERENCE_THRESHOLD} (executed once per image x anchor)")
print(f"Selected operating point : confidence={BEST_CONFIDENCE:.2f}, NMS IoU={BEST_NMS_IOU:.2f}")
print(f"Evaluation IoU           : {EVAL_IOU_THRESHOLD:.2f}")
print(f"Runs / images            : {len(runs)} runs over "
      f"{run_level_df['image_ID'].nunique()} images")
print("-" * 78)
print("EXPERIMENT-LEVEL RESULTS (mean over images, std between images)")
show = ["evaluation_mode", "AP50_mean", "AP50_std", "AP50_95_mean", "precision_mean",
        "recall_mean", "F1_mean", "F1_std", "IoU1_mean", "IoU2_mean"]
print(experiment_summary_df[show].to_string(index=False))
print("-" * 78)
print("POOLED DATASET AP")
print(dataset_ap_df[["evaluation_mode", "dataset_AP50", "dataset_AP50_95"]].to_string(index=False))
print("-" * 78)
print("POOLED CONFUSION COUNTS")
print(confusion_summary_df[["evaluation_mode", "TP", "FP", "FN",
                            "precision_micro", "recall_micro", "F1_micro"]].to_string(index=False))
print("=" * 78)

print("\nFiles written under", RESULTS_ROOT)
for root, dirs, files in os.walk(RESULTS_ROOT):
    depth = root.replace(RESULTS_ROOT, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    if os.path.basename(root) == "raw_detections":
        npz_files = [f for f in files if f.endswith(".npz")]
        for f in sorted(files):
            if not f.endswith(".npz"):
                print("  " * (depth + 1) + f)
        print("  " * (depth + 1) + f"[{len(npz_files)} run NPZ files]")
    else:
        for f in sorted(files):
            print("  " * (depth + 1) + f)
